## Compulsory Task 

In this compulsory task you will clean the country column and parse the date column in the **store_income_data_task.csv** file.

In [13]:
# import all necessary libraries
import pandas as pd
import fuzzywuzzy
from fuzzywuzzy import process
from datetime import datetime, date

# Load up store_income_data.csv
df = pd.read_csv("store_income_data_task.csv")

1. Take a look at all the unique values in the "country" column. Then, convert the column to lowercase and remove any trailing white spaces.

In [14]:
# Remove trailing white spaces and convert to lowercase to prevent an
# erroneous 'unique' entry. (Ex; UK and uk and Uk are the same)
df['country'] = df['country'].str.lower().str.strip()

# Determine the amount of unique countries and print using 'len'
unique_countries = df['country'].unique()
print(f"Unique countries: {len(unique_countries)}")

Unique countries: 37


2. Note that there should only be three separate countries. Eliminate all variations, so that 'South Africa', 'United Kingdom' and 'United States' are the only three countries.

In [15]:
# Example provided by 'data_cleaning_example.ipynb' provided a guideline for sorting
# Out the data entries and using a ratio to determine closeness. Then all rows with 
# a ratio match higher than 90 were replaced with the input match
def reduce_country_variations(df, column, desired_string, min_ratio = 90):
    
    strings = df[column].unique()

    matches = fuzzywuzzy.process.extract(desired_string, strings, 
                                         limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    near_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    matching_rows = df[column].isin(near_matches)
 
    df.loc[matching_rows, column] = desired_string

# Clean up any unwanted characters, as well as empty and nan cells
df['country'] = df['country'].str.replace('.', '').str.replace('/', '').str.replace('nan', '')
df.dropna(subset=['country'], inplace=True)
df = df[df['country'] != '']

# Make use of a dictionary to allow the addition of more entries
countries_to_replace = [
    "united kingdom", 
    "united states", 
    "united states of america", 
    "south africa", 
    "uk"
]

# Loop through the 'country_to_replace_' dictionary and apply it to the function
for country in countries_to_replace:
    reduce_country_variations(df, column='country', desired_string=country)


# Make use of a dictionary for column entries that need to be replaced, which allows an
# Entry like 'england' to be converted to UK. Making use of a dictionary would also 
# Make it easier add more 'differently named' countries easier
replacements = {
    'uk': 'united kingdom',
    'england': 'united kingdom',
    'britain': 'united kingdom',
    'america': 'united states',
    'sa': 'south africa',
    'united states of america': 'united states',
    's africasouth africa': 'south africa'
}

# Apply all replacements
df['country'] = df['country'].replace(replacements)

# Get all the unique values in the 'country' column
countries = df['country'].unique()

# Print the amount of unique countries using 'len'. Include an array 
print(f"There are {len(countries)} unique countries")
countries

There are 3 unique countries


array(['united states', 'united kingdom', 'south africa'], dtype=object)

3. Create a new column called `days_ago` in the DataFrame that is a copy of the 'date_measured' column but instead it is a number that shows how many days ago it was measured from the current date. Note that the current date can be obtained using `datetime.date.today()`.

In [16]:

# Convert the 'date_measured' column to datetime format. After dealing with copious amounts of
# Errors, 'errors='coerce' was found (Stackflow) and allowed to ignore invalid entries
df['date_measured'] = pd.to_datetime(df['date_measured'], errors='coerce')

# Get the current date, then determine the number of days ago
current_date = datetime.today()
df['days_ago'] = (current_date - df['date_measured']).dt.days

# Print the new column 'days_ago' as well as the first few entries
print(df[['date_measured', 'days_ago']].head())

  date_measured  days_ago
0    2006-04-02    6874.0
1    2006-04-01    6875.0
2    2003-12-09    7719.0
3    2006-08-05    6749.0
4           NaT       NaN
